In [ ]:
import random
import csv
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple



In [ ]:
@dataclass
class Restaurant:
    cuisines:list
    price_range:str
    has_self_del:bool
    has_offer:bool
    has_extra_del_cost:bool
    min_cost:float
    avg_rating:float
    avg_del_time:int
    payment_methods:list


In [ ]:


def generate_restaurants(
    rest_num: int = 100,
    cuisine_num_max: int = 4,
    cuisine_options: List[str] =  ['Pizza', 'Burger', 'Pasta', 'Souvlaki', 'Sushi', 'Chinese'],
    price_options: List[str] = ["$", "$$", "$$$", "$$$$"],
    price_options_distro: List[float] =  [0.5, 0.3, 0.15, 0.05],
    extra_del_cost_prob: float = 0.2,
    min_cost_gaussian_params: Tuple[float, float] = (6, 3),
    avg_rating_gaussian_params: Tuple[float, float] = (3, 1),
    avg_del_time_gaussian_params: Tuple[float, float] = (30, 15),
    payment_methods: Tuple[str, ...] = ('CASH', 'CARD', 'COUPON')
) -> List[Restaurant]:
    """
    Generates a list of restaurants with randomized attributes.

    Args:
    - rest_num (int): Number of restaurants to generate (default: 100).
    - cuisine_num_max (int): Maximum number of cuisines per restaurant (default: 4).
    - cuisine_options (List[str]): Available cuisines (default: ['Pizza', 'Burger', 'Pasta', 'Souvlaki', 'Sushi', 'Chinese']).
    - price_options (List[str]): Price range options (default: ["$", "$$", "$$$", "$$$$"]).
    - price_options_distro (List[float]): Probability distribution of price options (default: [0.5, 0.3, 0.15, 0.05]).
    - extra_del_cost_prob (float): Probability of extra delivery cost (default: 0.2).
    - min_cost_gaussian_params (Tuple[float, float]): Mean and std deviation for minimum order cost (default: (6,3)).
    - avg_rating_gaussian_params (Tuple[float, float]): Mean and std deviation for average rating (default: (3,1)).
    - avg_del_time_gaussian_params (Tuple[float, float]): Mean and std deviation for average delivery time (default: (30,15)).
    - payment_methods (Tuple[str, ...]): Available payment methods (default: ('CASH', 'CARD', 'COUPON')).

    Returns:
    - List[Restaurant]: A list of generated restaurant objects.
    """


    restaurants = []

    for _ in range(rest_num):
        # Pick a random number of cuisines
        cuisine_num = random.randint(1, cuisine_num_max)
        cuisines = random.sample(cuisine_options, cuisine_num)

        # Sample price range based on probability distribution
        price_index = np.random.choice(len(price_options), p=price_options_distro)
        price_range = price_options[price_index]

        # Random attributes
        has_self_del = random.choice([True, False])
        has_offer = random.choice([True, False])
        has_extra_del_cost = random.random() < extra_del_cost_prob

        # Sample Gaussian-distributed values (ensuring non-negative)
        min_cost = round(max(0, random.gauss(*min_cost_gaussian_params)), 1)
        avg_rating = round(max(0, random.gauss(*avg_rating_gaussian_params)), 1)
        avg_del_time = round(max(0, random.gauss(*avg_del_time_gaussian_params)), 1)

        # Sample payment methods
        selected_payment_methods = random.sample(payment_methods, random.randint(1, len(payment_methods)))

        # Create restaurant object
        restaurants.append(
            Restaurant(
                cuisines=cuisines,
                price_range=price_range,
                has_self_del=has_self_del,
                has_offer=has_offer,
                has_extra_del_cost=has_extra_del_cost,
                min_cost=min_cost,
                avg_rating=avg_rating,
                avg_del_time=avg_del_time,
                payment_methods=selected_payment_methods
            )
        )

    return restaurants


In [ ]:
restaurants=generate_restaurants(rest_num=100)

In [ ]:
restaurants[2]

Restaurant(cuisines=['Souvlaki', 'Sushi'], price_range='$$', has_self_del=True, has_offer=False, has_extra_del_cost=False, min_cost=9.1, avg_rating=2.0, avg_del_time=34.9, payment_methods=['CARD', 'COUPON', 'CASH'])

In [ ]:
@dataclass
class User:
    gender:str
    age:int
    favorite_cuisines:list


In [ ]:


def generate_users_segment1(user_num: int = 1000) -> List[User]:
    """
    Generates a list of users for Segment 1: One-Trick Pony.

    Characteristics:
    - 50% Male, 50% Female (randomly assigned)
    - Single favorite cuisine
    - Age follows a Gaussian distribution with a mean of 60 and std deviation of 7

    Args:
    - user_num (int): Number of users to generate (default: 1000)

    Returns:
    - List[User]: A list of generated users.
    """

    available_cuisines = ['Pizza', 'Burger', 'Pasta', 'Souvlaki', 'Sushi', 'Chinese']

    users = [
        User(
            gender=random.choice(["M", "F"]),
            age=max(0, int(random.gauss(60, 7))),  # Ensure non-negative age
            favorite_cuisines=random.sample(available_cuisines, 1)  # Single favorite cuisine
        )
        for _ in range(user_num)
    ]

    return users


In [ ]:

import csv
import random
from typing import List

def generate_ratings_segment1(users: List[User], restaurants: List[Restaurant], output_file: str = 'segment1.csv') -> None:
    """
    Generates ratings for Segment 1: One-Trick Pony.

    Criteria:
    - AVG_RATING > 4
    - AVG_DEL_TIME <= 35 min
    - HAS_OFFER: 60% influence on rating
    - 90% consistent rating behavior

    Args:
    - users (List[User]): List of user objects.
    - restaurants (List[Restaurant]): List of restaurant objects.
    - output_file (str): Name of the CSV file to save ratings (default: 'segment1.csv').

    Returns:
    - None (Writes data to CSV file).
    """

    with open(output_file, 'w', newline='') as fw:
        writer = csv.writer(fw)
        writer.writerow([
            'age', 'gender', 'favorite_cuisines', 'restaurant_cuisines', 'price_range',
            'has_self_del', 'has_offer', 'has_extra_del_cost', 'min_cost',
            'avg_rating', 'avg_del_time', 'payment_methods', 'rating', 'reason'
        ])

        positive_ratings = 0
        total_ratings = 0

        for user in users:
            rating_count = random.randint(1, len(restaurants))
            total_ratings += rating_count
            sampled_restaurants = random.sample(restaurants, rating_count)

            for restaurant in sampled_restaurants:
                rating = -1  # Default: negative rating
                reason = ""

                # Check if the restaurant matches user's single favorite cuisine and meets criteria
                if user.favorite_cuisines[0] in restaurant.cuisines:
                    if restaurant.avg_rating > 4 and restaurant.avg_del_time <= 35:
                        # 50-50 chance or boosted by HAS_OFFER (60% probability)
                        if random.random() < 0.5 or (restaurant.has_offer and random.random() <= 0.6):
                            rating = 1
                            positive_ratings += 1
                            reason = "Good rating & fast delivery"
                            if restaurant.has_offer:
                                reason += " + Offer available"
                        else:
                            reason = "Did not meet offer or random condition"
                    else:
                        reason = "Rating too low or delivery too slow"
                else:
                    reason = "Not preferred cuisine"

                # 90% consistency: 10% chance of flipping rating
                if random.random() < 0.1:
                    rating *= -1
                    reason += " Rating was flipped due to insconsistnet behavior"

                writer.writerow([
                    user.age, user.gender, user.favorite_cuisines, restaurant.cuisines, restaurant.price_range,
                    restaurant.has_self_del, restaurant.has_offer, restaurant.has_extra_del_cost,
                    restaurant.min_cost, restaurant.avg_rating, restaurant.avg_del_time,
                    restaurant.payment_methods, rating, reason
                ])

    print(f'Positive Ratings: {positive_ratings / total_ratings:.2%}')


In [ ]:


def generate_users_segment2(user_num: int = 1000, postcode_num: int = 50) -> List[User]:
    """
    Generates a list of users for Segment 2: young and price-driven.

    Characteristics:
    - 50% Male, 50% Female (randomly assigned)
    - Multiple favorite cuisines (at least 2)
    - Age follows a Gaussian distribution with a mean of 20 and std deviation of 7

    Args:
    - user_num (int): Number of users to generate (default: 1000)
    - postcode_num (int): Number of postcodes to generate (not used in function) (default: 50)

    Returns:
    - List[User]: A list of generated users.
    """

    available_cuisines = ['Pizza', 'Burger', 'Pasta', 'Souvlaki', 'Sushi', 'Chinese']

    users = [
        User(
            gender=random.choice(["M", "F"]),
            age=max(0, int(random.gauss(20, 7))),  # Ensure non-negative age
            favorite_cuisines=random.sample(available_cuisines, random.randint(2, 6))
        )
        for _ in range(user_num)
    ]

    return users


In [ ]:

def generate_ratings_segment2(users: List[User], restaurants: List[Restaurant], output_file: str = 'segment2.csv') -> None:
    """
    Generates ratings for Segment 2: young and price-driven.

    Criteria:
    - AVG_RATING > 4.2 if price is $$$
    - AVG_RATING > 3.5 if price is $$
    - AVG_RATING > 3 if price is $
    - No ratings if price is $$$$
    - No ratings if restaurant has an extra delivery cost
    - HAS_OFFER boosts rating probability: 80% influence for $$$, 60% influence for $, $$
    - 80% consistency in rating behavior

    Args:
    - users (List[User]): List of user objects.
    - restaurants (List[Restaurant]): List of restaurant objects.
    - output_file (str): Name of the CSV file to save ratings (default: 'segment2.csv').
    """

    with open(output_file, 'w', newline='') as fw:
        writer = csv.writer(fw)
        writer.writerow([
            'age', 'gender', 'favorite_cuisines', 'restaurant_cuisines', 'price_range',
            'has_self_del', 'has_offer', 'has_extra_del_cost', 'min_cost',
            'avg_rating', 'avg_del_time', 'payment_methods', 'rating', 'reason'
        ])

        positive_ratings = 0
        total_ratings = 0

        for user in users:
            num_ratings = random.randint(1, len(restaurants))
            total_ratings += num_ratings
            sampled_restaurants = random.sample(restaurants, num_ratings)

            for restaurant in sampled_restaurants:
                rating = -1  # Default negative rating
                reason = ""

                if any(cuisine in restaurant.cuisines for cuisine in user.favorite_cuisines):
                    if restaurant.has_extra_del_cost:
                        reason = "Extra delivery cost"
                    else:
                        if (
                            (restaurant.price_range == '$' and restaurant.avg_rating > 3) or
                            (restaurant.price_range == '$$' and restaurant.avg_rating > 3.5) or
                            (restaurant.price_range == '$$$' and restaurant.avg_rating > 4.2)
                        ):
                            # Rating decision logic with offer influence
                            if (
                                random.random() < 0.5 or
                                (restaurant.has_offer and restaurant.price_range in ['$', '$$'] and random.random() <= 0.6) or
                                (restaurant.has_offer and restaurant.price_range == '$$$' and random.random() <= 0.8)
                            ):
                                rating = 1
                                positive_ratings += 1
                                reason = "Good price-quality balance"
                                if restaurant.has_offer:
                                    reason += " + Offer available"
                            else:
                                reason = "Did not meet offer or price conditions"
                        else:
                            reason = "Did not meet rating and price conditions"
                else:
                    reason = "Not preferred cuisine"

                # 80% consistency factor (flip rating 20% of the time)
                if random.random() < 0.2:
                    rating *= -1
                    reason += " Inconsistent rating behavior"


                writer.writerow([
                    user.age, user.gender, user.favorite_cuisines, restaurant.cuisines, restaurant.price_range,
                    restaurant.has_self_del, restaurant.has_offer, restaurant.has_extra_del_cost,
                    restaurant.min_cost, restaurant.avg_rating, restaurant.avg_del_time,
                    restaurant.payment_methods, rating, reason
                ])

    print(f'Positive Ratings: {positive_ratings/total_ratings:.2%} ({positive_ratings}/{total_ratings})')



In [ ]:

users_segment1=generate_users_segment1(user_num=1000)
generate_ratings_segment1(users_segment1,restaurants)


Positive Ratings: 2.32%


In [ ]:


users_segment2=generate_users_segment2(user_num=2000)
generate_ratings_segment2(users_segment2,restaurants)


Positive Ratings: 14.41% (14491/100561)


In [ ]:
file_paths = ['segment1.csv', 'segment2.csv']  # Add as many paths as needed

# Read each CSV file into a DataFrame and store them in a list
dfs = [pd.read_csv(file_path) for file_path in file_paths]

# Concatenate all DataFrames into a single DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

# Shuffle the rows of the combined DataFrame
df = combined_df.sample(frac=1).reset_index(drop=True)

# shuffled_df now contains all your data, shuffled
print(df.shape)

(150871, 14)


In [ ]:
df

,age,gender,favorite_cuisines,restaurant_cuisines,price_range,has_self_del,has_offer,has_extra_del_cost,min_cost,avg_rating,avg_del_time,payment_methods,rating,reason
0,15,M,"['Pizza', 'Burger', 'Sushi']","['Pasta', 'Pizza', 'Chinese']",$$,True,False,False,4.6,2.9,40.9,"['CASH', 'COUPON', 'CARD']",-1,Did not meet rating and price conditions
1,9,M,"['Pizza', 'Sushi', 'Burger', 'Pasta', 'Souvlak...",['Souvlaki'],$$,False,True,False,7.4,1.7,57.6,['COUPON'],-1,Did not meet rating and price conditions
2,55,M,['Pizza'],"['Sushi', 'Pizza', 'Souvlaki', 'Chinese']",$,True,False,False,3.0,4.3,21.1,['CARD'],1,Did not meet offer or random condition Rating ...
3,27,F,"['Chinese', 'Souvlaki', 'Sushi', 'Burger', 'Pa...","['Pizza', 'Sushi']",$$$,True,True,False,4.0,1.2,17.7,['CARD'],1,Did not meet rating and price conditions Incon...
4,10,F,"['Souvlaki', 'Pizza', 'Pasta', 'Burger', 'Sushi']","['Pizza', 'Chinese', 'Souvlaki']",$$,True,True,False,7.8,2.0,46.3,['CASH'],-1,Did not meet rating and price conditions
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150866,60,M,['Burger'],['Burger'],$,True,False,False,5.9,3.6,18.0,"['CARD', 'COUPON']",-1,Rating too low or delivery too slow
150867,34,F,"['Burger', 'Chinese', 'Souvlaki', 'Pasta']","['Sushi', 'Chinese', 'Souvlaki', 'Pasta']",$$,True,True,False,4.8,2.1,14.8,"['CASH', 'COUPON', 'CARD']",-1,Did not meet rating and price conditions
150868,67,M,['Pizza'],"['Chinese', 'Burger']",$$,False,False,True,7.2,3.2,35.5,['COUPON'],-1,Not preferred cuisine
150869,55,F,['Burger'],"['Pizza', 'Chinese', 'Pasta']",$$,False,True,False,5.4,0.9,38.8,"['CASH', 'CARD']",-1,Not preferred cuisine


# Task
Preprocess the restaurant rating data in the `df` DataFrame by encoding categorical features like 'gender', 'cuisines', 'price_range', and 'payment_methods' into numerical formats. Split the processed dataset into training and testing sets, then train a classification model (such as Random Forest) to predict the 'rating' based on user and restaurant features. Evaluate the model's performance using a classification report or confusion matrix, and visualize feature importance to provide insights into the primary drivers of user ratings.

## Preprocess Data

### Subtask:
Clean and encode the categorical features of the `df` DataFrame into numerical formats suitable for machine learning.


**Reasoning**:
I will clean the dataframe by converting string-represented lists into usable formats, encoding ordinal and binary categorical variables, and dropping non-predictive columns to prepare the data for machine learning.



In [ ]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer

# 1. Handle string-represented lists using ast.literal_eval
list_cols = ['favorite_cuisines', 'restaurant_cuisines', 'payment_methods']
for col in list_cols:
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Multi-label encoding for cuisines and payment methods
mlb = MultiLabelBinarizer()

# Prefixing column names for clarity
def encode_multilabel(dataframe, column):
    encoded = mlb.fit_transform(dataframe[column])
    encoded_df = pd.DataFrame(encoded, columns=[f"{column}_{c}" for c in mlb.classes_])
    return encoded_df

fav_cuisines_df = encode_multilabel(df, 'favorite_cuisines')
rest_cuisines_df = encode_multilabel(df, 'restaurant_cuisines')
pay_methods_df = encode_multilabel(df, 'payment_methods')

# Combine and drop originals
df = pd.concat([df, fav_cuisines_df, rest_cuisines_df, pay_methods_df], axis=1)
df.drop(columns=list_cols, inplace=True)

# 2. Encode 'gender': M -> 0, F -> 1
df['gender'] = df['gender'].map({'M': 0, 'F': 1})

# 3. Encode 'price_range' ordinally
price_map = {'$': 1, '$$': 2, '$$$': 3, '$$$$': 4}
df['price_range'] = df['price_range'].map(price_map)

# 4. Convert 'rating' from [-1, 1] to [0, 1] for typical classification
df['rating'] = df['rating'].replace(-1, 0)

# 5. Drop non-predictive columns
df.drop(columns=['reason'], inplace=True)

# 6. Final verification
print("Data Types Check:\n", df.dtypes.value_counts())
print("\nNull Values Check:\n", df.isnull().sum().sum())
print(f"\nFinal Shape: {df.shape}")
df.head()

Data Types Check:
 int64      19
bool        3
float64     3
Name: count, dtype: int64

Null Values Check:
 0

Final Shape: (150871, 25)


,age,gender,price_range,has_self_del,has_offer,has_extra_del_cost,min_cost,avg_rating,avg_del_time,rating,...,favorite_cuisines_Sushi,restaurant_cuisines_Burger,restaurant_cuisines_Chinese,restaurant_cuisines_Pasta,restaurant_cuisines_Pizza,restaurant_cuisines_Souvlaki,restaurant_cuisines_Sushi,payment_methods_CARD,payment_methods_CASH,payment_methods_COUPON
0,15,0,2,True,False,False,4.6,2.9,40.9,0,...,1,0,1,1,1,0,0,1,1,1
1,9,0,2,False,True,False,7.4,1.7,57.6,0,...,1,0,0,0,0,1,0,0,0,1
2,55,0,1,True,False,False,3.0,4.3,21.1,1,...,0,0,1,0,1,1,1,1,0,0
3,27,1,3,True,True,False,4.0,1.2,17.7,1,...,1,0,0,0,1,0,1,1,0,0
4,10,1,2,True,True,False,7.8,2.0,46.3,0,...,1,0,1,0,1,1,0,0,1,0


## Split Dataset

### Subtask:
Divide the preprocessed `df` DataFrame into training and testing sets to prepare for model training and evaluation.


**Reasoning**:
I will split the preprocessed dataframe into feature matrix X and target vector y, then perform an 80/20 train-test split using scikit-learn to prepare for model training.



In [ ]:
from sklearn.model_selection import train_test_split

# 1. Define feature matrix X and target vector y
X = df.drop(columns=['rating'])
y = df['rating']

# 2. Split the data into training and testing sets (80/20 ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Verify the shapes
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Check class distribution to ensure stratification worked
print("\nTarget distribution in Train:\n", y_train.value_counts(normalize=True))
print("Target distribution in Test:\n", y_test.value_counts(normalize=True))

X_train shape: (120696, 24)
X_test shape: (30175, 24)
y_train shape: (120696,)
y_test shape: (30175,)

Target distribution in Train:
 rating
0    0.769678
1    0.230322
Name: proportion, dtype: float64
Target distribution in Test:
 rating
0    0.769677
1    0.230323
Name: proportion, dtype: float64


## Train Classifier

### Subtask:
Train a classification model, specifically a Random Forest, using the prepared training data to predict user ratings.


**Reasoning**:
I will import the RandomForestClassifier, instantiate it with the specified parameters, and train it on the training dataset as requested in the instructions.



In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. & 2. Instantiate the RandomForestClassifier with random_state and n_jobs
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)

# 3. Fit the classifier to the training data
rf_model.fit(X_train, y_train)

# 4. Model is stored in 'rf_model'. Verify it is ready.
print(f"Model trained successfully: {rf_model}")

Model trained successfully: RandomForestClassifier(n_jobs=-1, random_state=42)


In [ ]:
from sklearn.metrics import confusion_matrix,f1_score

In [ ]:
pred = rf_model.predict(X_test)

In [ ]:
confusion_matrix(y_test,pred)

array([[20221,  3004],
       [ 4620,  2330]])

In [ ]:
f1_score(y_test,pred)

0.37935525887333116